# T02 · Géométrie analytique — ADIF72

**Objectifs :**
- Normes (L1, L2, L∞, Minkowski) et distances associées
- Produit scalaire, orthogonalité, angles
- Projections et bases orthonormales (Gram-Schmidt)
- Similarité cosinus (applications : recherche, embeddings)

**Pourquoi c'est utile :**
| Besoin data science | Outil géométrique |
|---|---|
| Distance entre points (KNN, clustering) | L2, Manhattan, L∞ |
| Similarité (recherche, RAG, embeddings) | Cosinus, corrélation |
| Orthogonalisation (PCA, QR, SVD) | Gram-Schmidt |
| Reg (Lasso, ElasticNet) | Norme L1 (sparse) |


In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True, linewidth=100)
print("Prêt pour la géométrie. numpy", np.__version__)

Prêt pour la géométrie. numpy 2.5.0


## 1 · Normes d'un vecteur

Une **norme** est une fonction `‖·‖` qui mesure la « longueur » d'un vecteur :
1. `‖x‖ ≥ 0`, et `‖x‖ = 0 ⇔ x = 0`
2. `‖αx‖ = |α| ‖x‖` (homogénéité)
3. `‖x+y‖ ≤ ‖x‖ + ‖y‖` (inégalité triangulaire)

Les 3 normes usuelles :
| Norme | Formule | Usage typique |
|---|---|---|
| **L1** (Manhattan) | `Σ\|xᵢ\|` | Lasso (sparcity) |
| **L2** (Euclidienne) | `√(Σ xᵢ²)` | Moindres carrés |
| **L∞** (max) | `max\|xᵢ\|` | Bornes sup |

La **Minkowski** `‖x‖_p = (Σ \|xᵢ\|^p)^{1/p}` généralise L1 (p=1), L2 (p=2), L∞ (p→∞).

In [2]:
x = np.array([3.0, -4.0, 2.0])
print("x =", x)
print("L1      :", np.linalg.norm(x, ord=1))
print("L2      :", np.linalg.norm(x, ord=2))
print("L∞      :", np.linalg.norm(x, ord=np.inf))
print("Mink p=4:", np.linalg.norm(x, ord=4))

x = [ 3. -4.  2.]
L1      : 9.0
L2      : 5.385164807134504
L∞      : 4.0
Mink p=4: 4.334546600055874


## 2 · Produit scalaire, cosinus, angle

Le **produit scalaire** (dot product) de `u, v ∈ ℝⁿ` :
```
u·v = Σ uᵢ vᵢ  =  ‖u‖‖v‖ cos(θ)
```
C'est l'opération la plus importante en géométrie :
- `u·v = 0`  ⇔  **orthogonalité**
- `cos θ > 0`  ⇒  angle aigu (< 90°)
- `cos θ = 0`  ⇒  perpendiculaires
- `cos θ < 0`  ⇒  angle obtus (> 90°)

Le **produit vectoriel** en 3D donne un vecteur perpendiculaire à `u` et `v`
de norme `‖u‖‖v‖ sin θ` (direction = règle de la main droite).

In [3]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, -1.0, 2.0])
dot = u @ v
cos = dot / (np.linalg.norm(u) * np.linalg.norm(v))
angle = np.degrees(np.arccos(np.clip(cos, -1, 1)))
print(f"u·v = {dot}")
print(f"cos(θ) = {cos:.4f}")
print(f"θ = {angle:.2f}°")
# produit vectoriel
print("u × v =", np.cross(u, v))
assert np.isclose((np.cross(u, v)) @ u, 0)  # perpendiculaire à u
assert np.isclose((np.cross(u, v)) @ v, 0)  # et à v

u·v = 8.0
cos(θ) = 0.4666
θ = 62.19°
u × v = [ 7. 10. -9.]


## 3 · Orthogonalité & projection

**Projection orthogonale** de `u` sur `v` :
```
proj_v(u) = (u·v)/(v·v) · v
```
Le résidu `u - proj_v(u)` est **perpendiculaire** à `v`.
C'est la base de la géométrie des moindres carrés : on projette `y`
sur l'image de `X` pour trouver le `θ̂` qui minimise `‖Xθ - y‖`.

**Matrice de projection** sur un sous-espace `W` (spannés par les colonnes
d'une matrice `Q` de colonnes orthogonales) :
```
P_W = Q (Qᵀ Q)⁻¹ Qᵀ
```
Propriétés : `P² = P` (idempotente), `Pᵀ = P` (symétrique), `spec(P) ⊂ {0,1}`.

In [4]:
def proj(u, v):
    return (u @ v) / (v @ v) * v

u2 = np.array([2.0, 1.0]); v2 = np.array([1.0, 0.0])
p = proj(u2, v2)
print("proj_v(u) =", p)
print("Résidu = u - proj =", u2 - p)
assert np.isclose((u2 - p) @ v2, 0)   # perpendiculaire

# Matrice de projection sur le plan span(e1, e2) dans R3
Q = np.array([[1, 0], [0, 1], [0, 0]])   # plan xy
P = Q @ np.linalg.inv(Q.T @ Q) @ Q.T
print("\nProjection sur le plan xy (R^3 -> R^3) :\n", P)
v3 = np.array([1.0, 2.0, 5.0])
print("P @ [1,2,5] =", P @ v3, " (composante z annulée)")
assert np.allclose(P @ P, P)  # idempotente
assert np.allclose(P, P.T)    # symétrique

proj_v(u) = [2. 0.]
Résidu = u - proj = [0. 1.]

Projection sur le plan xy (R^3 -> R^3) :
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]]
P @ [1,2,5] = [1. 2. 0.]  (composante z annulée)


## 4 · Distances entre points

| Distance | Formule | Sens |
|---|---|---|
| **Euclidienne** | `‖a - b‖₂` | « ligne droite » (physique, géométrie) |
| **Manhattan / L1** | `Σ \|aᵢ-bᵢ\|` | « rue de la ville » (grille) |
| **Chebyshev / L∞** | `max \|aᵢ-bᵢ\|` | « roque d'échec » (mouvement 3D) |
| **Corrélation** | `1 - r(a,b)` | Indépendante à dérive/échelle (séries) |
| **Cosinus** | `1 - cos` | Indépendante à l'échelle (textes, embeddings) |

En clustering / KNN, le choix de la distance **change le résultat**.
La corrélation et le cosinus sont les plus robustes en data science.

In [5]:
a = np.array([1.0, 2.0, 3.0]); b = np.array([4.0, 6.0, 3.0])
print(f"Euclidienne  = {np.linalg.norm(a - b):.3f}")
print(f"Manhattan    = {np.abs(a - b).sum():.3f}")
print(f"Chebyshev    = {np.max(np.abs(a - b)):.3f}")

# Distance de corrélation (1 - cosinus des vectres centrés)
def dist_correlation(x, y):
    return 1 - np.corrcoef(x, y)[0, 1]

xs = np.array([1.0, 2, 3, 4, 5])
ys = np.array([1.1, 2.2, 2.9, 4.1, 4.8])
print(f"Corrélation  = {dist_correlation(xs, ys):.5f} (0 = parfaitement corrélées)")

Euclidienne  = 5.000
Manhattan    = 7.000
Chebyshev    = 4.000
Corrélation  = 0.00339 (0 = parfaitement corrélées)


## 5 · Bases orthonormales — Gram-Schmidt

Une **base orthonormale** : des vecteurs de norme 1 mutuellement
perpendiculaires. C'est la « meilleure » base pour :
- diagonaliser une matrice symétrique (théorème spectral)
- décomposer une matrice en QR (`Q` orthogonale, `R` triangulaire)
- le SVD (`U` et `V` sont orthonormales)

**Algorithme de Gram-Schmidt** : orthonormalise une base quelconque
en soustrayant, un à un, la composante projetée sur les précédents.

Variantes :
- **Classique** (ce que nous implémentons) — instable numériquement si les
  vecteurs sont presque parallèles.
- **Modifiée** (MGS) — plus stable, c'est celle qu'utilise `scipy.linalg.qr`.

In [6]:
def gram_schmidt(V):
    """Orthonormalise les colonnes de V."""
    Q = []
    for v in V.T:
        q = v.copy()
        for w in Q:
            q = q - (q @ w) * w
        n = np.linalg.norm(q)
        if n < 1e-12:
            continue
        q = q / n
        Q.append(q)
    return np.stack(Q).T

W = np.array([[1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
Qm = gram_schmidt(W)
print("Colonnes d'origine :\n", W)
print("Base orthonormale Q :\n", Qm)
print("Q.T @ Q ≈ I ?", np.allclose(Qm.T @ Qm, np.eye(2), atol=1e-8))

# Comparaison avec la QR de numpy
Qq, Rq = np.linalg.qr(W)
print("\nQR numpy : Q=\n", Qq, " (signes peuvent différer, mêmes directions)")

Colonnes d'origine :
 [[1. 1.]
 [0. 1.]
 [1. 0.]]
Base orthonormale Q :
 [[ 0.707  0.408]
 [ 0.     0.816]
 [ 0.707 -0.408]]
Q.T @ Q ≈ I ? True

QR numpy : Q=
 [[-0.707 -0.408]
 [-0.    -0.816]
 [-0.707  0.408]]  (signes peuvent différer, mêmes directions)


## 6 · Application : similarité cosinus (textes, embeddings)

Le **cosinus de similarité** mesure l'angle entre deux vecteurs, **indépendant
de leur longueur**. C'est la métrique reine pour :
- la recherche sémantique
- les embeddings (word2vec, BERT, etc.)
- le clustering de documents
- la détection de plagiat

`cos = 1` : mêmes directions (max similarité) ; `cos = 0` : perpendiculaires ;
`cos = -1` : opposés.

In [7]:
def cosine_sim(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

# fr/mots simples : TF (term frequency)
doc1 = np.array([3, 1, 2, 0])   # [chat, chien, lapin, oiseau]
doc2 = np.array([2, 1, 3, 0])
doc3 = np.array([0, 0, 5, 4])
print(f"sim(doc1, doc2) = {cosine_sim(doc1, doc2):.3f}")
print(f"sim(doc1, doc3) = {cosine_sim(doc1, doc3):.3f}")
assert cosine_sim(doc1, doc2) > cosine_sim(doc1, doc3)

# Invariance à l'échelle : multiplier doc1 par 100 ne change pas le cosinus
print(f"sim(100*doc1, doc2) = {cosine_sim(100*doc1, doc2):.3f} (invariant !)")

sim(doc1, doc2) = 0.929
sim(doc1, doc3) = 0.417
sim(100*doc1, doc2) = 0.929 (invariant !)


## 🎯 Exercices corrigés


In [8]:
# Ex 1 : normes de a=(5,-1,2)
a1 = np.array([5.0, -1.0, 2.0])
print(f"Ex1 : L1={np.linalg.norm(a1,1)}, L2={np.linalg.norm(a1,2):.3f}, Linf={np.linalg.norm(a1,np.inf)}")
assert np.linalg.norm(a1, 1) == 8

Ex1 : L1=8.0, L2=5.477, Linf=5.0


In [9]:
# Ex 2 : angle entre p=(1,0,0) et q=(0,1,0)
p = np.array([1.0, 0, 0]); q = np.array([0.0, 1, 0])
ang = np.degrees(np.arccos(p @ q / (np.linalg.norm(p) * np.linalg.norm(q))))
print(f"Ex2 : angle(p,q) = {ang}° (orthogonaux)")
assert ang == 90

Ex2 : angle(p,q) = 90.0° (orthogonaux)


In [10]:
# Ex 3 : projection de m=(3,4) sur n=(1,2)
# coef = m·n / n·n = (3+8)/(1+4) = 11/5 = 2.2  ->  proj = 2.2*n = (2.2, 4.4)
m = np.array([3.0, 4.0]); n = np.array([1.0, 2.0])
coef = (m @ n) / (n @ n)
pr = coef * n
print(f"Ex3 : coef = {coef}, proj = {pr}")
assert np.allclose(pr, [2.2, 4.4])
assert np.isclose((m - pr) @ n, 0)   # résidu perpendiculaire

Ex3 : coef = 2.2, proj = [2.2 4.4]


In [11]:
# Ex 4 : distance euclidienne entre (0,0,0) et (1,1,1)
d = np.linalg.norm(np.array([1.0, 1, 1]))
print(f"Ex4 : dist = {d:.4f} = √3")
assert np.isclose(d, np.sqrt(3))

Ex4 : dist = 1.7321 = √3


In [12]:
# Ex 5 : Gram-Schmidt sur [[1,1],[1,0]]
Vs = np.array([[1.0, 1.0], [1.0, 0.0]])
Qs = gram_schmidt(Vs)
print("Ex5 : Q =\n", Qs)
print("Q.T @ Q =\n", Qs.T @ Qs)
assert np.allclose(Qs.T @ Qs, np.eye(2), atol=1e-8)

print("\n✅ T02 terminé.")

Ex5 : Q =
 [[ 0.707  0.707]
 [ 0.707 -0.707]]
Q.T @ Q =
 [[1. 0.]
 [0. 1.]]

✅ T02 terminé.
